# HIPPIE tutorial: embedding and cross-species classification

HIPPIE is a conditional VAE that learns technology-adjusted representations of
extracellular recordings from three modalities: the mean **waveform (WF)**, the
**inter-spike-interval (ISI)** distribution, and the **autocorrelogram (ACG)**.

This notebook is a minimal, runnable tour of the public `hippie.inference` API:

1. load the cross-dataset pretrained checkpoint,
2. preprocess and embed a dataset,
3. classify cell types with a KNN probe (balanced accuracy, the paper's primary metric),
4. visualize the embedding with UMAP,
5. transfer across species in HIPPIE's shared latent space.

Everything runs on datasets shipped under `datasets_hippie/` plus the public checkpoint;
no training and no GPU required.

## 1. Setup

```bash
pip install -e ".[viz]"   # the [viz] extra adds umap-learn, used for the plots
```

In [ ]:
import os, re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from hippie.inference import HIPPIEClassifier, TECHNOLOGY_IDS, select_k_via_cv

# Locate datasets_hippie regardless of the kernel's working directory.
# VS Code often runs a notebook from the workspace root, not the repo root,
# so we search upward and also look inside a HIPPIE/ subfolder.
def _find_dir(name='datasets_hippie', max_up=6):
    here = os.path.abspath(os.getcwd())
    for k in range(max_up + 1):
        base = os.path.abspath(os.path.join(here, *(['..'] * k)))
        for cand in (os.path.join(base, name), os.path.join(base, 'HIPPIE', name)):
            if os.path.isdir(cand):
                return cand
    raise FileNotFoundError(
        f"Could not find '{name}'. Open the HIPPIE folder as your VS Code "
        f"workspace, or set this path to an absolute location.")
DATASETS_ROOT = _find_dir()
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
print('TECHNOLOGY_IDS:', TECHNOLOGY_IDS)

## 2. Load the pretrained model

The checkpoint is pulled from the HuggingFace Hub (`Jesusgf23/hippie`). It was pretrained
jointly across the study's datasets, so no per-dataset training is needed here. To use a
local file instead: `HIPPIEClassifier.from_checkpoint('path/to/model.ckpt', device=DEVICE)`.

In [ ]:
classifier = HIPPIEClassifier.from_pretrained(device=DEVICE)
print(f'Model loaded on {DEVICE}.')

## 3. Preprocessing the three modalities

Each modality is resampled to a fixed length and min-max normalized to `[-1, 1]`:
WF -> 50 points, ISI -> 100 bins (after a `log(x+1)` transform), ACG -> 100 bins.

> **ACG NaN gotcha.** Some units have an all-zero (constant) ACG row. A naive
> `(x - min) / (max - min)` divides by zero on those rows and yields `NaN`, which then
> propagates through BatchNorm and poisons *every* embedding in the batch. We guard the
> denominator with an epsilon and call `nan_to_num`, so constant rows normalize to zeros rather than NaN. Datasets that ship no
> `acg.csv` are zero-filled and run in bimodal mode.

In [ ]:
def _resample_minmax(raw, target_len):
    """Resample each row to target_len and min-max normalize to [-1, 1] (NaN-safe)."""
    t = torch.as_tensor(raw, dtype=torch.float32)
    if t.dim() == 1:
        t = t.unsqueeze(0)
    if t.shape[-1] != target_len:
        t = F.interpolate(t.unsqueeze(1), size=(target_len,), mode='linear',
                          align_corners=False).squeeze(1)
    mn = t.amin(dim=-1, keepdim=True)
    mx = t.amax(dim=-1, keepdim=True)
    out = (t - mn) / (mx - mn + 1e-8) * 2.0 - 1.0   # +eps: constant rows -> 0, not NaN
    return torch.nan_to_num(out).numpy().astype(np.float32)

def preprocess_wave(arr): return _resample_minmax(arr, 50)
def preprocess_acg(arr):  return _resample_minmax(np.nan_to_num(arr), 100)
def preprocess_isi(arr):  return _resample_minmax(np.log(np.nan_to_num(arr) + 1.0), 100)

## 4. Cleaning the labels

The shipped `labels.csv` files store cell types as **byte-strings** (e.g. `b'PkC_ss'`) and
include unlabeled rows (`b''`, `b'unlabeled'`). Read naively, pandas keeps the literal text
`"b'PkC_ss'"` and the unlabeled rows stay in. We strip the wrapper and drop unlabeled rows so
the classifier never trains on empty labels.

In [ ]:
_BYTESTR = re.compile(r"""^b(['\"])(.*)\1$""")

def clean_labels(raw):
    """Strip the b'...' byte-string wrapper; return (labels, keep_mask).
    keep_mask is False for empty / 'unlabeled' rows."""
    cleaned = np.array([
        (_BYTESTR.match(s).group(2) if _BYTESTR.match(s) else s)
        for s in raw.astype(str)
    ])
    keep = ~np.isin(cleaned, ['', 'unlabeled', 'nan', 'None'])
    return cleaned, keep

## 5. Load a dataset

`load_dataset` reads the four CSVs, cleans labels, drops unlabeled rows, removes rare cell
types (fewer than `min_per_class` units, as in the paper's Hull preprocessing), and preprocesses
the modalities. If a dataset ships no `acg.csv`, HIPPIE runs in bimodal mode with a zero ACG.

In [ ]:
def load_dataset(name, root=DATASETS_ROOT, min_per_class=5):
    d = os.path.join(root, name)
    wf  = pd.read_csv(os.path.join(d, 'waveforms.csv')).to_numpy().astype(np.float32)
    isi = pd.read_csv(os.path.join(d, 'isi_dist.csv')).to_numpy().astype(np.float32)
    acg_path = os.path.join(d, 'acg.csv')
    if os.path.exists(acg_path):
        acg = pd.read_csv(acg_path).to_numpy().astype(np.float32)
    else:
        print(f'  [{name}] no acg.csv -> bimodal mode (zero-filled ACG)')
        acg = np.zeros((len(wf), 100), dtype=np.float32)

    labels, keep = clean_labels(pd.read_csv(os.path.join(d, 'labels.csv')).iloc[:, -1])
    wf, isi, acg, labels = wf[keep], isi[keep], acg[keep], labels[keep]

    # drop rare classes (the paper applies N<5 removal on Hull)
    types, counts = np.unique(labels, return_counts=True)
    common = set(types[counts >= min_per_class])
    m = np.isin(labels, list(common))
    wf, isi, acg, labels = wf[m], isi[m], acg[m], labels[m]

    out = dict(wave=preprocess_wave(wf), isi=preprocess_isi(isi),
               acg=preprocess_acg(acg), labels=labels)
    assert np.isfinite(out['acg']).all(), 'ACG contains NaN/Inf after preprocessing!'
    return out

hull = load_dataset('hull_cell_type')   # mouse cerebellar neurons (Neuropixels)
print(f"Hull: {len(hull['labels'])} neurons, classes {sorted(set(hull['labels']))}")

## 6. Extract embeddings and classify (KNN probe)

`tech_id` conditions the encoder on the recording technology (Hull = Neuropixels). We then
evaluate with a cosine-distance KNN on L2-normalized embeddings: `k` is chosen by
cross-validation on the training split (`select_k_via_cv`), and we report **balanced
accuracy** (recall averaged over classes), the paper's primary metric for imbalanced data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score

def l2(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

emb_hull = classifier.get_embeddings(
    wave=hull['wave'], isi=hull['isi'], acg=hull['acg'], tech_id='neuropixels')
print('embeddings:', emb_hull.shape, '(N x z_dim)')

X_tr, X_te, y_tr, y_te = train_test_split(
    emb_hull, hull['labels'], test_size=0.2, stratify=hull['labels'], random_state=42)
best_k, _ = select_k_via_cv(X_tr, y_tr)
knn = KNeighborsClassifier(n_neighbors=best_k, metric='cosine').fit(l2(X_tr), y_tr)
acc = balanced_accuracy_score(y_te, knn.predict(l2(X_te)))
print(f'best k = {best_k}')
print(f'balanced accuracy = {acc:.3f}  (chance = {1/len(set(hull["labels"])):.3f})')

## 7. Visualize the embedding (UMAP)

In [ ]:
import matplotlib.pyplot as plt

coords = classifier.umap_reduce(emb_hull, n_components=2)
plt.figure(figsize=(6, 5))
for ct in sorted(set(hull['labels'])):
    m = hull['labels'] == ct
    plt.scatter(coords[m, 0], coords[m, 1], s=10, label=ct)
plt.legend(markerscale=2, fontsize=8, loc='best')
plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2')
plt.title('HIPPIE embedding - Hull (mouse cerebellum)')
plt.tight_layout(); plt.show()

## 8. Cross-species transfer in the shared latent space

A single pretrained model embeds every dataset into one latent space, so we can train a
classifier on one species and apply it to another. Here we train on **macaque** cerebellum
(Lisberger) and predict **mouse** (Hull), over the four cerebellar cell types shared by both
(`PkC_ss, PkC_cs, MLI, MFB`; chance = 0.25).

> **Source-conditioning note.** The encoder is conditioned on recording technology, so each
> dataset's embeddings carry a per-source offset. To compare neurons *across* datasets we
> encode them under a single common technology token, which puts both species in the same
> reference frame (the inference-time analogue of HIPPIE's source conditioning). Skipping this
> step leaves the two species in separate regions and the transfer collapses to chance.

This is a simplified illustration of the shared latent space, not a reproduction of the
paper's benchmark. The paper's cross-species analysis emphasizes the shared latent geometry
and generative transfer over raw classification rank, and its reported cross-species numbers
come from a dedicated cross-dataset-pretrained model (with a 3D-ACG variant). Expect an
above-chance result here, but do not read it as a paper number.

In [ ]:
from matplotlib.lines import Line2D

SHARED   = ['PkC_ss', 'PkC_cs', 'MLI', 'MFB']   # cerebellar types shared by both datasets
REF_TECH = 'neuropixels'                         # common reference technology for comparison

def restrict(ds, classes):
    m = np.isin(ds['labels'], classes)
    return {k: v[m] for k, v in ds.items()}

macaque = restrict(load_dataset('lisberger_labeled_cell_type'), SHARED)
mouse   = restrict(hull, SHARED)

# Encode both under a common technology token so the source-conditioning offset is shared.
emb_mac   = classifier.get_embeddings(wave=macaque['wave'], isi=macaque['isi'],
                                      acg=macaque['acg'], tech_id=REF_TECH)
emb_mouse = classifier.get_embeddings(wave=mouse['wave'], isi=mouse['isi'],
                                      acg=mouse['acg'], tech_id=REF_TECH)

# (a) Transfer: train on macaque, predict mouse.
best_k, _ = select_k_via_cv(emb_mac, macaque['labels'])
knn = KNeighborsClassifier(n_neighbors=best_k, metric='cosine').fit(l2(emb_mac), macaque['labels'])
acc = balanced_accuracy_score(mouse['labels'], knn.predict(l2(emb_mouse)))
print(f'train macaque (Lisberger) n={len(emb_mac)} -> test mouse (Hull) n={len(emb_mouse)}')
print(f'cross-species balanced accuracy = {acc:.3f}  (chance = 0.25)')

# (b) Visualize both species in the shared latent space.
emb_all = np.vstack([emb_mac, emb_mouse])
lab_all = np.concatenate([macaque['labels'], mouse['labels']])
sp_all  = np.array(['macaque'] * len(emb_mac) + ['mouse'] * len(emb_mouse))
coords  = classifier.umap_reduce(emb_all, n_components=2)

colors = dict(zip(SHARED, plt.cm.tab10.colors))
plt.figure(figsize=(6, 5))
for ct in SHARED:
    for sp, marker in [('macaque', 'o'), ('mouse', '^')]:
        m = (lab_all == ct) & (sp_all == sp)
        plt.scatter(coords[m, 0], coords[m, 1], s=20, marker=marker,
                    color=colors[ct], edgecolor='k', linewidth=0.2)
handles  = [Line2D([], [], marker='s', linestyle='', color=colors[ct], label=ct) for ct in SHARED]
handles += [Line2D([], [], marker='o', linestyle='', color='grey', label='macaque'),
            Line2D([], [], marker='^', linestyle='', color='grey', label='mouse')]
plt.legend(handles=handles, fontsize=7, loc='best')
plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2')
plt.title('Shared latent space: macaque vs mouse cerebellum')
plt.tight_layout(); plt.show()

## Next steps

- **Your own data:** drop `{waveforms,isi_dist,acg,labels}.csv` into a new
  `datasets_hippie/<name>/` folder and call `load_dataset('<name>')`. Columns of any length
  are resampled automatically; set the `tech_id` to the matching recording hardware.
- **Batch extraction** across many datasets: see `examples/extract_embeddings.py`.
- **Clustering / web app:** `HIPPIEClassifier` also exposes `hdbscan_cluster(...)`.